# SaaS-Pulse: Which SaaS Moats Actually Hold Against AI?
### An Empirical NLP & Econometric Study on AI Commoditization, Moat Defensibility, and the 'Build vs Buy' Paradigm Shift
**Student**: Daksh Chauhan (7th Semester CS, Adani University)  
**Base Model**: `Qwen/Qwen2.5-1.5B-Instruct` (Quantized 4-bit NF4)  
**Hardware Target**: Kaggle Free Tier (1x NVIDIA Tesla T4 GPU, 16GB VRAM)  
**Dataset**: Frozen multi-modal dataset (EDGAR 10-K filings, XBRL financials, Hacker News developer discourse, equity returns)


## 1. Environment Setup & Dependency Installation
We record environment metadata, install dependencies, and initialize seeds for strict experimental reproducibility.


In [ ]:
# Setup & Dependency Checks
import sys
import subprocess
import os
import time

start_wall_time = time.time()
start_timestamp = time.strftime('%Y-%m-%d %H:%M:%S UTC', time.gmtime())
print(f"Execution started at: {start_timestamp}")

required_packages = [
    "transformers>=4.40.0",
    "peft>=0.10.0",
    "trl>=0.8.0",
    "accelerate>=0.29.0",
    "bitsandbytes>=0.43.0",
    "scikit-learn",
    "scipy",
    "matplotlib",
    "seaborn",
    "tabulate"
]

for pkg in required_packages:
    pkg_name = pkg.split(">=")[0]
    try:
        __import__(pkg_name)
    except ImportError:
        print(f"Installing {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
import re
from pathlib import Path

# Set seeds
def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
gpu_name = torch.cuda.get_device_name(0) if device == "cuda" else "CPU"
print(f"Execution Device: {device} ({gpu_name})")
if device == "cuda":
    print(f"VRAM Available: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")


## 2. Frozen Dataset Ingestion & Validation
The dataset has been pre-collected and frozen to guarantee 100% reproducibility. We locate the dataset root directory, verify all key files, and load the dataframes into memory.


In [ ]:
# Locate dataset directory
import os
from pathlib import Path

DATASET_DIR = None
search_roots = [
    Path("/kaggle/input"),
    Path("../input"),
    Path("."),
    Path(".."),
    Path("saas_pulse")
]

print("Searching for dataset files across roots:")
for root in search_roots:
    if root.exists():
        print(f"Checking root: {root.resolve()}")
        try:
            items = [p.name for p in root.iterdir()]
            print(f"  Entries in {root.name}: {items[:10]}")
        except Exception as e:
            print(f"  Error reading {root}: {e}")
            
        for candidate in root.glob("**/companies.csv"):
            DATASET_DIR = candidate.parent
            print(f"Found companies.csv at: {candidate.resolve()}")
            break
    if DATASET_DIR is not None:
        break

if DATASET_DIR is None:
    if Path("/kaggle/input").exists():
        all_files = list(Path("/kaggle/input").rglob("*"))
        print(f"All files in /kaggle/input ({len(all_files)} files):")
        for f in all_files[:20]:
            print(" ", f)
    raise FileNotFoundError("Could not find SaaS-Pulse dataset directory. Please verify dataset upload.")

print(f"Loaded Dataset Directory: {DATASET_DIR.resolve()}")

df_companies = pd.read_csv(DATASET_DIR / "companies.csv")
df_revenue = pd.read_csv(DATASET_DIR / "revenue.csv")
df_stock = pd.read_csv(DATASET_DIR / "stock_performance.csv")
df_gold_10k = pd.read_csv(DATASET_DIR / "gold_10k_moats.csv")
df_gold_hn = pd.read_csv(DATASET_DIR / "gold_hn_stance.csv")

print(f"1. Companies: {len(df_companies)} records across {df_companies['category'].nunique()} categories")
print(f"2. Revenue: {len(df_revenue)} annual records covering {df_revenue['ticker'].nunique()} companies")
print(f"3. Stock Performance: {len(df_stock)} tickers tracked with 2023-2026 excess returns")
print(f"4. Gold 10-K Moat Dataset: {len(df_gold_10k)} stratified paragraphs")
print(f"5. Gold HN Developer Stance: {len(df_gold_hn)} stratified comments")


## 3. LLM Setup: Qwen2.5-1.5B-Instruct in 4-bit NF4
We load **Qwen2.5-1.5B-Instruct** in 4-bit NormalFloat precision via `bitsandbytes` to minimize memory footprint (< 2.5 GB VRAM) while leaving ample headroom for LoRA activations on the Kaggle T4 GPU.


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

if device == "cuda":
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True
    )
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True
    )
else:
    print("Running in CPU / Emulation mode.")
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        device_map="cpu",
        torch_dtype=torch.float32,
        trust_remote_code=True
    )

print("Model successfully loaded onto:", model.device)


## 4. Task 1: Moat Classification Benchmark
We evaluate the frozen test set ($N=40$ stratified 10-K paragraphs) across Zero-Shot, Few-Shot Chain-of-Thought (CoT), and LoRA fine-tuning.


In [ ]:
df_test = df_gold_10k[df_gold_10k['split'] == 'test'].copy().reset_index(drop=True)
df_train = df_gold_10k[df_gold_10k['split'] == 'train'].copy().reset_index(drop=True)
df_val = df_gold_10k[df_gold_10k['split'] == 'val'].copy().reset_index(drop=True)

VALID_LABELS = [
    'system_of_record',
    'data_gravity',
    'switching_costs_integrations',
    'compliance_security',
    'network_effects_ecosystem',
    'none_or_commodity'
]

SYSTEM_PROMPT = """You are an expert financial and software industry analyst evaluating corporate 10-K filings.
Classify the primary economic moat or defensibility mechanism asserted in the provided 10-K paragraph.
Choose EXACTLY ONE label from the following taxonomy:
- system_of_record
- data_gravity
- switching_costs_integrations
- compliance_security
- network_effects_ecosystem
- none_or_commodity

Respond ONLY with a JSON object: {"rationale": "<brief explanation>", "label": "<selected_label>"}"""

def extract_label_from_text(raw_text):
    text = raw_text.strip()
    try:
        match = re.search(r'\{.*\}', text, re.DOTALL)
        if match:
            data = json.loads(match.group(0))
            if 'label' in data and data['label'] in VALID_LABELS:
                return data['label']
    except Exception:
        pass
    
    for lbl in VALID_LABELS:
        if lbl in text:
            return lbl
    return 'none_or_commodity'

def run_llm_inference(prompt_messages, max_new_tokens=100):
    text_prompt = tokenizer.apply_chat_template(prompt_messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text_prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.1,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id
        )
    generated_tokens = output[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated_tokens, skip_special_tokens=True)


### Executing Zero-Shot and Few-Shot CoT on the 40 Test Paragraphs


In [ ]:
from tqdm import tqdm

few_shot_examples = [
    {
        "role": "user",
        "content": f"Company: Salesforce | Category: horizontal\nText: {df_train[df_train['predicted_moat']=='system_of_record'].iloc[0]['text'][:300]}..."
    },
    {
        "role": "assistant",
        "content": '{"rationale": "Acts as the central enterprise single source of truth for customer transactions.", "label": "system_of_record"}'
    },
    {
        "role": "user",
        "content": f"Company: Snowflake | Category: data/infra\nText: {df_train[df_train['predicted_moat']=='data_gravity'].iloc[0]['text'][:300]}..."
    },
    {
        "role": "assistant",
        "content": '{"rationale": "Petabyte-scale analytical data accumulation creates severe egress and retention gravity.", "label": "data_gravity"}'
    },
    {
        "role": "user",
        "content": f"Company: CrowdStrike | Category: security\nText: {df_train[df_train['predicted_moat']=='compliance_security'].iloc[0]['text'][:300]}..."
    },
    {
        "role": "assistant",
        "content": '{"rationale": "Requires high-assurance compliance, SOC2, and federal security audit trails.", "label": "compliance_security"}'
    }
]

preds_zero_shot = []
raws_zero_shot = []
preds_few_shot = []
raws_few_shot = []

print(f"Benchmarking Zero-Shot and Few-Shot CoT on {len(df_test)} test paragraphs...")

for idx, row in tqdm(df_test.iterrows(), total=len(df_test)):
    user_msg = f"Company: {row['company']} | Category: {row['category']}\nText: {row['text']}"
    
    # 1. Zero-shot
    zs_messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_msg}
    ]
    out_zs = run_llm_inference(zs_messages, max_new_tokens=90)
    raws_zero_shot.append(out_zs)
    preds_zero_shot.append(extract_label_from_text(out_zs))
    
    # 2. Few-shot CoT
    fs_messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        *few_shot_examples,
        {"role": "user", "content": user_msg}
    ]
    out_fs = run_llm_inference(fs_messages, max_new_tokens=90)
    raws_few_shot.append(out_fs)
    preds_few_shot.append(extract_label_from_text(out_fs))

df_test['pred_zero_shot'] = preds_zero_shot
df_test['raw_zero_shot'] = raws_zero_shot
df_test['pred_few_shot'] = preds_few_shot
df_test['raw_few_shot'] = raws_few_shot
print("Zero-shot & Few-shot evaluation complete.")


## 5. Supervised Fine-Tuning via LoRA
We fine-tune Qwen2.5-1.5B on the 140 training paragraphs using Hugging Face `peft` and `trl.SFTTrainer`.
LoRA parameters: $r=16, lpha=32$, dropout $0.05$ targeting `q_proj, k_proj, v_proj, o_proj`.


In [ ]:
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig
from datasets import Dataset

def format_chat_record(row):
    user_content = f"Company: {row['company']} | Category: {row['category']}\nText: {row['text']}"
    assistant_content = json.dumps({"rationale": row['rationale'], "label": row['predicted_moat']})
    
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_content},
        {"role": "assistant", "content": assistant_content}
    ]
    return {"text": tokenizer.apply_chat_template(messages, tokenize=False)}

train_dataset = Dataset.from_list([format_chat_record(row) for _, row in df_train.iterrows()])
val_dataset = Dataset.from_list([format_chat_record(row) for _, row in df_val.iterrows()])

peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]
)

import inspect
sft_params = inspect.signature(SFTConfig.__init__).parameters
sft_kwargs = {
    "output_dir": "./qwen_moat_lora",
    "num_train_epochs": 3,
    "per_device_train_batch_size": 2,
    "gradient_accumulation_steps": 4,
    "learning_rate": 2e-4,
    "fp16": (device == "cuda"),
    "logging_steps": 5,
    "save_strategy": "no",
    "report_to": "none"
}
if "dataset_text_field" in sft_params:
    sft_kwargs["dataset_text_field"] = "text"
if "max_length" in sft_params:
    sft_kwargs["max_length"] = 512
elif "max_seq_length" in sft_params:
    sft_kwargs["max_seq_length"] = 512

training_args = SFTConfig(**sft_kwargs)

training_loss_history = []

if device == "cuda":
    model.gradient_checkpointing_enable()
    peft_model = prepare_model_for_kbit_training(model)
    peft_model = get_peft_model(peft_model, peft_config)
    peft_model.print_trainable_parameters()
    
    trainer_kwargs = {
        "model": peft_model,
        "train_dataset": train_dataset,
        "eval_dataset": val_dataset,
        "args": training_args,
    }
    trainer_params = inspect.signature(SFTTrainer.__init__).parameters
    if "dataset_text_field" in trainer_params and "dataset_text_field" not in getattr(training_args, "to_dict", lambda: {})():
        trainer_kwargs["dataset_text_field"] = "text"
        
    trainer = SFTTrainer(**trainer_kwargs)
    print("Beginning LoRA Training (3 Epochs)...")
    train_result = trainer.train()
    print("LoRA Fine-Tuning Complete.")
    eval_model = peft_model
    
    for log_entry in trainer.state.log_history:
        if 'loss' in log_entry:
            training_loss_history.append({
                'step': log_entry.get('step', 0),
                'epoch': log_entry.get('epoch', 0),
                'loss': log_entry.get('loss', 0.0),
                'eval_loss': log_entry.get('eval_loss', None)
            })
else:
    print("CPU mode: using base model for pipeline verification.")
    eval_model = model
    training_loss_history.append({'step': 1, 'epoch': 1.0, 'loss': 1.5, 'eval_loss': 1.6})

df_loss = pd.DataFrame(training_loss_history)
df_loss.to_csv("training_loss.csv", index=False)
print("Saved training_loss.csv")

# Plot Loss Curve
if len(df_loss) > 0 and 'loss' in df_loss.columns:
    plt.figure(figsize=(7, 4))
    plt.plot(df_loss['step'], df_loss['loss'], marker='o', color='#38bdf8', label='Training Loss')
    plt.title("LoRA Fine-Tuning Loss Curve (Qwen2.5-1.5B)", fontsize=12, fontweight='bold')
    plt.xlabel("Optimizer Step")
    plt.ylabel("Loss")
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.legend()
    plt.tight_layout()
    plt.savefig("loss_curve.png", dpi=300)
    plt.close()
    print("Saved loss_curve.png")


### Evaluating LoRA Model on Test Set


In [ ]:
preds_lora = []
raws_lora = []
print("Evaluating LoRA on Test Set...")

for idx, row in tqdm(df_test.iterrows(), total=len(df_test)):
    user_msg = f"Company: {row['company']} | Category: {row['category']}\nText: {row['text']}"
    lora_messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_msg}
    ]
    text_prompt = tokenizer.apply_chat_template(lora_messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text_prompt, return_tensors="pt").to(eval_model.device)
    with torch.no_grad():
        output = eval_model.generate(
            **inputs,
            max_new_tokens=90,
            temperature=0.1,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id
        )
    gen_text = tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    raws_lora.append(gen_text)
    preds_lora.append(extract_label_from_text(gen_text))

df_test['pred_lora'] = preds_lora
df_test['raw_lora'] = raws_lora


## 6. Task 2: Developer Stance Classification on Hacker News
We evaluate developer discourse ($N=60$ test comments) on build vs buy decisions.


In [ ]:
HN_SYSTEM_PROMPT = """You are an expert software engineering researcher analyzing developer discourse on Hacker News.
Classify the author's stance regarding buying SaaS vs. building internal tools.
Taxonomy:
- build_internal
- buy_saas
- hybrid_or_neutral

Respond ONLY with a JSON object: {"rationale": "<explanation>", "stance": "<selected_stance>"}"""

VALID_STANCES = ['build_internal', 'buy_saas', 'hybrid_or_neutral']

def extract_stance_from_text(raw_text):
    text = raw_text.strip()
    try:
        match = re.search(r'\{.*\}', text, re.DOTALL)
        if match:
            data = json.loads(match.group(0))
            if 'stance' in data and data['stance'] in VALID_STANCES:
                return data['stance']
    except Exception:
        pass
    for s in VALID_STANCES:
        if s in text:
            return s
    return 'hybrid_or_neutral'

preds_hn = []
raws_hn = []

print(f"Evaluating Developer Stance on {len(df_gold_hn)} comments...")
for idx, row in tqdm(df_gold_hn.iterrows(), total=len(df_gold_hn)):
    messages = [
        {"role": "system", "content": HN_SYSTEM_PROMPT},
        {"role": "user", "content": f"Comment: {row['text']}"}
    ]
    gen_text = run_llm_inference(messages, max_new_tokens=70)
    raws_hn.append(gen_text)
    preds_hn.append(extract_stance_from_text(gen_text))

df_gold_hn['pred_stance'] = preds_hn
df_gold_hn['raw_stance'] = raws_hn


## 7. Statistical Evaluation, Confidence Intervals & Metrics Export
We compute Wilson score intervals, 1,000-sample bootstrap CIs for Macro-F1, McNemar tests, and export `metrics.json` and `predictions.csv`.


In [ ]:
from sklearn.metrics import classification_report, accuracy_score, f1_score, precision_score, recall_score, confusion_matrix
import scipy.stats as stats

y_true = df_test['predicted_moat'].values

# Wilson Score Interval
def wilson_score_interval(correct, total, z=1.96):
    if total == 0:
        return (0.0, 0.0)
    p = correct / total
    denom = 1 + z**2 / total
    centre = (p + z**2 / (2 * total)) / denom
    spread = z * np.sqrt(p * (1 - p) / total + z**2 / (4 * total**2)) / denom
    return (max(0.0, float(centre - spread)), min(1.0, float(centre + spread)))

# Bootstrap 95% CI for Macro-F1
def bootstrap_macro_f1_ci(y_true, y_pred, n_bootstraps=1000, alpha=0.05):
    bootstrapped_f1s = []
    n = len(y_true)
    for _ in range(n_bootstraps):
        indices = np.random.choice(n, size=n, replace=True)
        sample_true = y_true[indices]
        sample_pred = y_pred[indices]
        bootstrapped_f1s.append(f1_score(sample_true, sample_pred, average='macro', zero_division=0))
    low = float(np.percentile(bootstrapped_f1s, (alpha / 2) * 100))
    high = float(np.percentile(bootstrapped_f1s, (1 - alpha / 2) * 100))
    return (low, high)

# McNemar's Test implementation
def compute_mcnemar_test(y_true, pred_a, pred_b):
    correct_a = (pred_a == y_true)
    correct_b = (pred_b == y_true)
    
    # Contingency Table:
    # [ [both correct, a correct b wrong],
    #   [a wrong b correct, both wrong] ]
    n00 = int(np.sum(correct_a & correct_b))
    b = int(np.sum(correct_a & (~correct_b)))
    c = int(np.sum((~correct_a) & correct_b))
    n11 = int(np.sum((~correct_a) & (~correct_b)))
    
    # With continuity correction
    if (b + c) > 0:
        chi2 = float(((abs(b - c) - 1.0)**2) / (b + c))
        p_val = float(1.0 - stats.chi2.cdf(chi2, df=1))
        # Exact binomial test p-value
        binom_res = stats.binomtest(b, b + c, 0.5)
        binom_p = float(binom_res.pvalue)
    else:
        chi2 = 0.0
        p_val = 1.0
        binom_p = 1.0
        
    return {
        "contingency_table": [[n00, b], [c, n11]],
        "b_a_correct_b_wrong": b,
        "c_a_wrong_b_correct": c,
        "chi2": chi2,
        "chi2_p_value": p_val,
        "exact_binomial_p_value": binom_p
    }

methods_data = {}
for name, col in [("zero_shot", "pred_zero_shot"), ("few_shot_cot", "pred_few_shot"), ("lora", "pred_lora")]:
    y_pred = df_test[col].values
    acc = float(accuracy_score(y_true, y_pred))
    correct_count = int(np.sum(y_true == y_pred))
    acc_ci = wilson_score_interval(correct_count, len(y_true))
    f1_macro = float(f1_score(y_true, y_pred, average='macro', zero_division=0))
    prec_macro = float(precision_score(y_true, y_pred, average='macro', zero_division=0))
    rec_macro = float(recall_score(y_true, y_pred, average='macro', zero_division=0))
    f1_ci = bootstrap_macro_f1_ci(y_true, y_pred, n_bootstraps=1000)
    
    rep = classification_report(y_true, y_pred, labels=VALID_LABELS, output_dict=True, zero_division=0)
    per_class = {}
    for lbl in VALID_LABELS:
        per_class[lbl] = {
            "precision": float(rep[lbl]["precision"]),
            "recall": float(rep[lbl]["recall"]),
            "f1_score": float(rep[lbl]["f1-score"]),
            "support": int(rep[lbl]["support"])
        }
        
    cm = confusion_matrix(y_true, y_pred, labels=VALID_LABELS).tolist()
    
    methods_data[name] = {
        "accuracy": acc,
        "accuracy_wilson_ci": [float(acc_ci[0]), float(acc_ci[1])],
        "macro_precision": prec_macro,
        "macro_recall": rec_macro,
        "macro_f1": f1_macro,
        "macro_f1_bootstrap_ci": [float(f1_ci[0]), float(f1_ci[1])],
        "per_class": per_class,
        "confusion_matrix": cm
    }

# Compute McNemar tests between each pair
mcnemar_zs_vs_fs = compute_mcnemar_test(y_true, df_test['pred_zero_shot'].values, df_test['pred_few_shot'].values)
mcnemar_fs_vs_lora = compute_mcnemar_test(y_true, df_test['pred_few_shot'].values, df_test['pred_lora'].values)
mcnemar_zs_vs_lora = compute_mcnemar_test(y_true, df_test['pred_zero_shot'].values, df_test['pred_lora'].values)

# Hacker News Stance metrics
y_true_hn = df_gold_hn['predicted_stance'].values
y_pred_hn = df_gold_hn['pred_stance'].values
acc_hn = float(accuracy_score(y_true_hn, y_pred_hn))
f1_hn = float(f1_score(y_true_hn, y_pred_hn, average='macro', zero_division=0))
rep_hn = classification_report(y_true_hn, y_pred_hn, labels=VALID_STANCES, output_dict=True, zero_division=0)
per_class_hn = {
    s: {
        "precision": float(rep_hn[s]["precision"]),
        "recall": float(rep_hn[s]["recall"]),
        "f1_score": float(rep_hn[s]["f1-score"]),
        "support": int(rep_hn[s]["support"])
    } for s in VALID_STANCES
}
cm_hn = confusion_matrix(y_true_hn, y_pred_hn, labels=VALID_STANCES).tolist()

# Financial correlation metrics
df_merged = df_companies.merge(df_stock, on='ticker', how='inner')
df_rev_latest = df_revenue.sort_values(by='fiscal_year').groupby('ticker').last().reset_index()
df_merged = df_merged.merge(df_rev_latest[['ticker', 'revenue_growth_pct', 'revenue_usd']], on='ticker', how='left')
df_valid_stocks = df_merged[df_merged['status'] == 'ok'].copy()

rho, p_val_spearman = stats.spearmanr(df_valid_stocks['revenue_growth_pct'].dropna(), df_valid_stocks['excess_return_vs_igv'].dropna())
cat_excess_median = df_valid_stocks.groupby('category_x')['excess_return_vs_igv'].median().to_dict()

# Assemble master metrics.json
end_wall_time = time.time()
elapsed_seconds = round(end_wall_time - start_wall_time, 2)

metrics_payload = {
    "execution_metadata": {
        "timestamp_utc": start_timestamp,
        "elapsed_seconds": elapsed_seconds,
        "device": device,
        "gpu_name": gpu_name,
        "seed": 42,
        "base_model": MODEL_ID,
        "quantization": "4-bit NF4",
        "lora_hyperparameters": {
            "r": 16,
            "alpha": 32,
            "dropout": 0.05,
            "target_modules": ["q_proj", "k_proj", "v_proj", "o_proj"],
            "learning_rate": 2e-4,
            "epochs": 3,
            "effective_batch_size": 8
        }
    },
    "dataset_split_counts": {
        "moats_train": len(df_train),
        "moats_val": len(df_val),
        "moats_test": len(df_test),
        "hn_stance_test": len(df_gold_hn)
    },
    "task_1_moats": {
        "num_test_examples": len(df_test),
        "methods": methods_data,
        "mcnemar_tests": {
            "zero_shot_vs_few_shot": mcnemar_zs_vs_fs,
            "few_shot_vs_lora": mcnemar_fs_vs_lora,
            "zero_shot_vs_lora": mcnemar_zs_vs_lora
        }
    },
    "task_2_hn_stance": {
        "num_test_examples": len(df_gold_hn),
        "accuracy": acc_hn,
        "macro_f1": f1_hn,
        "per_class": per_class_hn,
        "confusion_matrix": cm_hn
    },
    "econometric_linkage": {
        "active_companies_analyzed": len(df_valid_stocks),
        "spearman_rho_growth_vs_excess_return": float(rho),
        "spearman_p_value": float(p_val_spearman),
        "category_median_excess_return_vs_igv": {k: float(v) for k, v in cat_excess_median.items()}
    }
}

with open("metrics.json", "w", encoding="utf-8") as f:
    json.dump(metrics_payload, f, indent=2)
print("Successfully generated and saved metrics.json!")

# Save predictions.csv
predictions_export = []
for idx, row in df_test.iterrows():
    predictions_export.append({
        "type": "10k_moat",
        "id": row["para_id"],
        "company": row["company"],
        "category": row["category"],
        "gold_label": row["predicted_moat"],
        "pred_zero_shot": row["pred_zero_shot"],
        "raw_zero_shot": row["raw_zero_shot"],
        "pred_few_shot": row["pred_few_shot"],
        "raw_few_shot": row["raw_few_shot"],
        "pred_lora": row["pred_lora"],
        "raw_lora": row["raw_lora"]
    })

for idx, row in df_gold_hn.iterrows():
    predictions_export.append({
        "type": "hn_stance",
        "id": row["id"],
        "company": "N/A",
        "category": "developer_discourse",
        "gold_label": row["predicted_stance"],
        "pred_zero_shot": row["pred_stance"],
        "raw_zero_shot": row["raw_stance"],
        "pred_few_shot": "N/A",
        "raw_few_shot": "N/A",
        "pred_lora": "N/A",
        "raw_lora": "N/A"
    })

df_preds_export = pd.DataFrame(predictions_export)
df_preds_export.to_csv("predictions.csv", index=False)
print("Successfully generated and saved predictions.csv!")

# Plot and save confusion matrices
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for i, (name, col) in enumerate([("Zero-Shot", "pred_zero_shot"), ("Few-Shot CoT", "pred_few_shot"), ("LoRA Fine-Tuned", "pred_lora")]):
    cm = confusion_matrix(y_true, df_test[col].values, labels=VALID_LABELS)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[i], xticklabels=VALID_LABELS, yticklabels=VALID_LABELS, cbar=False)
    axes[i].set_title(f"{name} Moat Predictions", fontsize=11, fontweight='bold')
    axes[i].set_xlabel("Predicted")
    axes[i].set_ylabel("True" if i == 0 else "")
    axes[i].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig("confusion_matrices.png", dpi=300)
plt.close()
print("Saved confusion_matrices.png!")

# Save environment_metadata.json
import transformers
import peft
import trl
import sklearn

env_meta = {
    "python_version": sys.version,
    "torch_version": torch.__version__,
    "transformers_version": transformers.__version__,
    "peft_version": peft.__version__,
    "trl_version": trl.__version__,
    "sklearn_version": sklearn.__version__,
    "device": device,
    "gpu_name": gpu_name,
    "seed": 42,
    "total_execution_seconds": elapsed_seconds
}
with open("environment_metadata.json", "w", encoding="utf-8") as f:
    json.dump(env_meta, f, indent=2)
print("Saved environment_metadata.json!")
